<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Учёт заказов товаров и услуг

Вариант задания № 9. 



<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Продолжение проекта учёта заказов из Sprint1, вариант 9. Программа хранит товары, сведения о клиенте и статус заказа, рассчитывает суммы и скидки, поддерживает перенос позиций, планирование доставки и оформление подарков. Онлайн-заказы содержат сведения об оплате и отслеживании. Экспресс-заказ дополняет онлайн-заказ сроком доставки. Отслеживание и вывод уведомлений представлены отдельными интерфейсами.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;

interface ITrackable
{
    string TrackingNumber { get; set; }
    void Track();
}

interface INotifiable
{
    void NotifyCustomer();
}

class Item
{
    private decimal price = 0m;
    private int quantity = 1;

    public string Name { get; }

    public decimal Price
    {
        get { return price; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException("price", "Цена не может быть отрицательной.");

            price = value;
        }
    }

    public int Quantity
    {
        get { return quantity; }
        set
        {
            if (value <= 0)
                throw new ArgumentOutOfRangeException("quantity", "Количество должно быть положительным.");

            quantity = value;
        }
    }

    public Item(string name, decimal price = 0m, int quantity = 1)
    {
        if (string.IsNullOrWhiteSpace(name))
            throw new ArgumentException("Название позиции не должно быть пустым.", "name");

        Name = name;
        Price = price;
        Quantity = quantity;
    }

    public decimal GetCost() { return Price * Quantity; }
}

abstract class Order
{
    private List<Item> items = new List<Item>();

    public int OrderId { get; }
    public DateTime CreationDate { get; }
    public decimal TotalAmount { get; protected set; }
    public int ItemCount { get { return items.Count; } }

    public string CustomerName { get; set; } = "Не указан";
    public string Status { get; protected set; } = "Новый";
    public string Comment { get; set; } = "Нет";

    protected Order(int orderId, DateTime creationDate)
    {
        if (orderId <= 0)
            throw new ArgumentOutOfRangeException("orderId", "Номер заказа должен быть положительным.");

        OrderId = orderId;
        CreationDate = creationDate;
    }

    public void ConfirmOrder() { Status = "Подтверждён"; }
    public void CancelOrder() { Status = "Отменён"; }

    public void ShowOrderInfo()
    {
        CalculateTotal();
        Console.WriteLine($"Заказ #{OrderId}, дата {CreationDate:dd.MM.yyyy}, позиций: {ItemCount}, сумма: {TotalAmount:F2} руб.");
        ShowItems();
        Console.WriteLine($"Клиент: {CustomerName}, статус: {Status}, комментарий: {Comment}.");
    }

    public virtual decimal CalculateTotal()
    {
        TotalAmount = 0;
        foreach (Item item in items)
            TotalAmount += item.GetCost();

        return TotalAmount;
    }

    public virtual void AddItem(Item item)
    {
        if (item == null)
            throw new ArgumentNullException("item");

        items.Add(item);
        CalculateTotal();
    }

    public virtual bool RemoveItem(Item item)
    {
        if (item == null)
            throw new ArgumentNullException("item");

        bool removed = items.Remove(item);
        if (removed)
            CalculateTotal();

        return removed;
    }

    public bool TransferItemTo(Order targetOrder, Item item)
    {
        if (targetOrder == null)
            throw new ArgumentNullException("targetOrder");

        if (RemoveItem(item))
        {
            targetOrder.AddItem(item);
            return true;
        }

        return false;
    }

    public void ShowItems()
    {
        foreach (Item item in items)
            Console.WriteLine($"  {item.Name}: {item.Price:F2} руб. x {item.Quantity} = {item.GetCost():F2} руб.");
    }
}

class OnlineOrder : Order, ITrackable, INotifiable
{
    public string CustomerEmail { get; }
    public string DeliveryMethod { get; }

    public string Website { get; set; } = "shop.example";
    public string TrackingNumber { get; set; } = "Не присвоен";
    public bool IsPaid { get; private set; } = false;

    public OnlineOrder(int orderId, DateTime creationDate,
                       string customerEmail, string deliveryMethod)
        : base(orderId, creationDate)
    {
        if (string.IsNullOrWhiteSpace(customerEmail))
            throw new ArgumentException("Email не должен быть пустым.", "customerEmail");
        if (string.IsNullOrWhiteSpace(deliveryMethod))
            throw new ArgumentException("Укажите способ доставки.", "deliveryMethod");

        CustomerEmail = customerEmail;
        DeliveryMethod = deliveryMethod;
    }

    public override void AddItem(Item item)
    {
        base.AddItem(item);
        Console.WriteLine($"Онлайн-заказ #{OrderId}: добавлено «{item.Name}»; доставка — {DeliveryMethod}.");
    }

    public void ShowDeliveryInfo()
    {
        Console.WriteLine($"Заказ #{OrderId}: {DeliveryMethod}; email клиента: {CustomerEmail}.");
    }

    public void MarkAsPaid()
    {
        CalculateTotal();
        IsPaid = true;
        Console.WriteLine($"Заказ #{OrderId}: оплата отмечена: {IsPaid}, сумма: {TotalAmount:F2} руб.");
    }

    public void Track()
    {
        Console.WriteLine($"Отслеживание #{OrderId}: сайт {Website}, номер {TrackingNumber}.");
    }

    public void NotifyCustomer()
    {
        Console.WriteLine($"Уведомление для {CustomerEmail}: заказ #{OrderId}, статус: {Status}.");
    }
}

class PhysicalOrder : Order
{
    public string DeliveryAddress { get; }

    public string CourierName { get; private set; } = "Не назначен";
    public DateTime DeliveryDate { get; set; }
    public bool IsDelivered { get; private set; } = false;

    public PhysicalOrder(int orderId, DateTime creationDate, string deliveryAddress)
        : base(orderId, creationDate)
    {
        if (string.IsNullOrWhiteSpace(deliveryAddress))
            throw new ArgumentException("Адрес доставки не должен быть пустым.", "deliveryAddress");

        DeliveryAddress = deliveryAddress;
        DeliveryDate = creationDate;
    }

    public override bool RemoveItem(Item item)
    {
        bool removed = base.RemoveItem(item);
        if (removed)
            Console.WriteLine($"Физический заказ #{OrderId}: возврат «{item.Name}» зарегистрирован; позиция удалена.");
        else
            Console.WriteLine($"Физический заказ #{OrderId}: позиции «{item.Name}» нет; возврат не оформлен.");

        return removed;
    }

    public void ShowDeliveryAddress()
    {
        Console.WriteLine($"Адрес доставки заказа #{OrderId}: {DeliveryAddress}.");
    }

    public void AssignCourier(string courierName) { CourierName = courierName; }

    public void ShowDeliveryPlan()
    {
        Console.WriteLine($"Доставка #{OrderId}: {DeliveryDate:dd.MM.yyyy}, курьер: {CourierName}, адрес: {DeliveryAddress}.");
    }

    public void MarkAsDelivered()
    {
        IsDelivered = true;
        Status = "Доставлен";
        Console.WriteLine($"Заказ #{OrderId}: доставка завершена: {IsDelivered}.");
    }
}

class SpecializedOrder : Order
{
    public string SpecialConditions { get; }
    public decimal DiscountPercent { get; }

    public string ManagerName { get; private set; } = "Не назначен";
    public string GiftMessage { get; private set; } = "Нет";
    public bool IsGift { get; private set; } = false;

    public SpecializedOrder(int orderId, DateTime creationDate,
                            string specialConditions, decimal discountPercent)
        : base(orderId, creationDate)
    {
        if (string.IsNullOrWhiteSpace(specialConditions))
            throw new ArgumentException("Укажите специальные условия.", "specialConditions");
        if (discountPercent < 0 || discountPercent > 100)
            throw new ArgumentOutOfRangeException("discountPercent", "Скидка должна быть от 0 до 100 процентов.");

        SpecialConditions = specialConditions;
        DiscountPercent = discountPercent;
    }

    public override decimal CalculateTotal()
    {
        TotalAmount = decimal.Round(base.CalculateTotal() * (1 - DiscountPercent / 100m),
                                    2, MidpointRounding.AwayFromZero);
        return TotalAmount;
    }

    public void ShowSpecialConditions()
    {
        Console.WriteLine($"Условия заказа #{OrderId}: {SpecialConditions}; скидка {DiscountPercent}%.");
    }

    public void AssignManager(string managerName) { ManagerName = managerName; }

    public void MakeGift(string message)
    {
        IsGift = true;
        GiftMessage = message;
    }

    public void ShowGiftInfo()
    {
        Console.WriteLine($"Заказ #{OrderId}: менеджер: {ManagerName}, подарок: {IsGift}, пожелание: {GiftMessage}");
    }
}

class ExpressOrder : OnlineOrder
{
    public int DeliveryHours { get; }

    public ExpressOrder(int orderId, DateTime creationDate, string customerEmail,
                        string deliveryMethod, int deliveryHours)
        : base(orderId, creationDate, customerEmail, deliveryMethod)
    {
        if (deliveryHours <= 0)
            throw new ArgumentOutOfRangeException("deliveryHours", "Срок доставки должен быть положительным.");

        DeliveryHours = deliveryHours;
    }

    public void ShowDeadline()
    {
        Console.WriteLine($"Экспресс-заказ #{OrderId}: доставка в течение {DeliveryHours} ч.");
    }
}

DateTime creationDate = new DateTime(2026, 10, 20);

OnlineOrder onlineOrder1 = new OnlineOrder(101, creationDate, "buyer1@example.com", "курьер");
OnlineOrder onlineOrder2 = new OnlineOrder(102, creationDate, "buyer2@example.com", "пункт выдачи");
PhysicalOrder physicalOrder1 = new PhysicalOrder(201, creationDate, "Учебная улица, дом 1");
PhysicalOrder physicalOrder2 = new PhysicalOrder(202, creationDate, "Учебная улица, дом 2");
SpecializedOrder specializedOrder1 = new SpecializedOrder(301, creationDate, "Скидка постоянному клиенту", 10m);
SpecializedOrder specializedOrder2 = new SpecializedOrder(302, creationDate, "Скидка на услуги", 20m);
ExpressOrder expressOrder = new ExpressOrder(401, new DateTime(2026, 10, 20),
                                           "express@example.com", "экспресс-курьер", 6);

Order[] orders = {
    onlineOrder1, onlineOrder2, physicalOrder1, physicalOrder2,
    specializedOrder1, specializedOrder2, expressOrder
};

foreach (Order order in orders)
{
    order.CustomerName = "Учебный клиент";
    order.Comment = "Позвонить перед доставкой";
    order.ConfirmOrder();
}

Console.WriteLine("Добавление позиций:");
onlineOrder1.AddItem(new Item("Мышь", 750m, 2));
Item keyboard = new Item("Клавиатура", 2000m);
onlineOrder1.AddItem(keyboard);
onlineOrder2.AddItem(new Item("Флешка", 600m, 3));

Item monitor = new Item("Монитор", 12000m);
physicalOrder1.AddItem(monitor);
Item cable = new Item("Кабель", 500m, 2);
physicalOrder1.AddItem(cable);
physicalOrder2.AddItem(new Item("Наушники", 2500m));
specializedOrder1.AddItem(new Item("Комплект мебели", 20000m));
specializedOrder2.AddItem(new Item("Установка оборудования", 5000m, 2));
expressOrder.AddItem(new Item("Веб-камера", 3000m));

Console.WriteLine();
Console.WriteLine("Удаление позиции и информация о возврате:");
physicalOrder1.RemoveItem(monitor);
physicalOrder1.RemoveItem(monitor);

Console.WriteLine();
Console.WriteLine("Перенос позиции между заказами:");
bool transferred = onlineOrder1.TransferItemTo(onlineOrder2, keyboard);
Console.WriteLine($"Позиция «{keyboard.Name}» перенесена из заказа #101 в заказ #102: {transferred}.");

Console.WriteLine();
Console.WriteLine("Изменение количества позиции:");
cable.Quantity = 3;
physicalOrder1.CalculateTotal();
Console.WriteLine($"Кабель: {cable.Quantity} шт.; сумма заказа #201: {physicalOrder1.TotalAmount:F2} руб.");

Console.WriteLine();
Console.WriteLine("Дополнительная информация:");
onlineOrder1.ShowDeliveryInfo();
onlineOrder2.ShowDeliveryInfo();
physicalOrder1.ShowDeliveryAddress();
physicalOrder2.ShowDeliveryAddress();
specializedOrder1.ShowSpecialConditions();
specializedOrder2.ShowSpecialConditions();

Console.WriteLine();
Console.WriteLine("Новые действия заказов:");
onlineOrder1.TrackingNumber = "TRK-101";
onlineOrder1.MarkAsPaid();
onlineOrder1.Track();
onlineOrder1.NotifyCustomer();
onlineOrder2.CancelOrder();
onlineOrder2.NotifyCustomer();

physicalOrder1.DeliveryDate = new DateTime(2026, 10, 20);
physicalOrder1.AssignCourier("Сергей");
physicalOrder1.ShowDeliveryPlan();
physicalOrder1.MarkAsDelivered();

specializedOrder1.AssignManager("Анна");
specializedOrder1.MakeGift("С новосельем!");
specializedOrder1.ShowGiftInfo();

expressOrder.TrackingNumber = "EXP-401";
expressOrder.MarkAsPaid();
expressOrder.Track();
expressOrder.NotifyCustomer();
expressOrder.ShowDeadline();

Console.WriteLine();
Console.WriteLine("Состав и суммы заказов:");
foreach (Order order in orders)
    order.ShowOrderInfo();


Добавление позиций:
Онлайн-заказ #101: добавлено «Мышь»; доставка — курьер.
Онлайн-заказ #101: добавлено «Клавиатура»; доставка — курьер.
Онлайн-заказ #102: добавлено «Флешка»; доставка — пункт выдачи.
Онлайн-заказ #401: добавлено «Веб-камера»; доставка — экспресс-курьер.

Удаление позиции и информация о возврате:
Физический заказ #201: возврат «Монитор» зарегистрирован; позиция удалена.
Физический заказ #201: позиции «Монитор» нет; возврат не оформлен.

Перенос позиции между заказами:
Онлайн-заказ #102: добавлено «Клавиатура»; доставка — пункт выдачи.
Позиция «Клавиатура» перенесена из заказа #101 в заказ #102: True.

Изменение количества позиции:
Кабель: 3 шт.; сумма заказа #201: 1500,00 руб.

Дополнительная информация:
Заказ #101: курьер; email клиента: buyer1@example.com.
Заказ #102: пункт выдачи; email клиента: buyer2@example.com.
Адрес доставки заказа #201: Учебная улица, дом 1.
Адрес доставки заказа #202: Учебная улица, дом 2.
Условия заказа #301: Скидка постоянному клиенту; ски